#  C++ Primer : Compile-time Definitions
# 1. `#define` and `-D`
* Before the compiler proper sees your file, a _preprocessor_ runs over it and performs textual substitutions
* `#define N 1000` makes the preprocessor replace every later occurrence of the name `N` by `1000`
* The same thing can be done from the command line, without touching the source:
```
clang++ -Wall -DN=1000 prog.cc -o prog
```
* The two are equivalent; `-D` just lets you decide at compile time
* The customary idiom is to give the name a default in the source and let `-D` override it
```
#ifndef N
#define N 1000
#endif
```
  * `#ifndef N` means "if `N` has not been defined, use the following lines"; `-DN=...` on the command line defines it first and the default is skipped

# 2. Why bother
* A value known at compile time can be used where a run-time value cannot: as the size of a plain array, or **as the size of an array inside a `struct`**, which is the main reason this course uses `-D`
```
struct point_set {
  long n;
  double x[N];       /* N comes from -DN=... */
  double y[N];
};
```
  * the size of the type itself now depends on how you compiled it. One source file, a family of data structures
* The compiler can also take advantage of a known value --- unrolling a loop whose trip count it knows, for example --- which is why performance experiments in this course often compare different `-D` values rather than different command line arguments
* The cost is that each value needs its own compilation, and the name is _not_ a variable: it is text, substituted before the language is even parsed. `#define TWICE(x) x * 2` and then `TWICE(1 + 1)` expands to `1 + 1 * 2`, which is 3


In [ ]:
%%writefile defs.cc
#include <stdio.h>

#ifndef N
#define N 10
#endif

struct point_set {
  long n;
  double x[N];
  double y[N];
};

int main() {
  point_set s;
  s.n = N;
  for (long i = 0; i < s.n; i++) {
    s.x[i] = i;
    s.y[i] = i * i;
  }
  printf("N = %d, sizeof(point_set) = %ld, y[N-1] = %f\n",
         N, sizeof(point_set), s.y[N - 1]);
  return 0;
}

* Compile it twice, with two different values, into two different executables


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall defs.cc -o defs_default
clang++ -Wall -DN=100 defs.cc -o defs_100

In [ ]:
%%bash
./defs_default
./defs_100

* <font color="green">Exercise: (a) the source was never edited and yet the two executables behave differently --- explain in one sentence what `-DN=100` did to the text the compiler saw. (b) try `-DN=10+10` and predict, before running it, what `sizeof(point_set)` and `y[N-1]` will be; run it and explain. (c) add a second compile-time name `ELEM` with default `double`, compile with `-DELEM=float`, and compare `sizeof(point_set)`.</font>

* Start from this copy


In [ ]:
%%writefile defs_ex.cc
#include <stdio.h>

#ifndef N
#define N 10
#endif

/* (c) give ELEM a default here, and use it as the element type below */

struct point_set {
  long n;
  double x[N];
  double y[N];
};

int main() {
  point_set s;
  s.n = N;
  for (long i = 0; i < s.n; i++) {
    s.x[i] = i;
    s.y[i] = i * i;
  }
  printf("N = %d, sizeof(point_set) = %ld, y[N-1] = %f\n",
         N, sizeof(point_set), (double)s.y[N - 1]);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall defs_ex.cc -o defs_ex
# clang++ -Wall -DN=10+10 defs_ex.cc -o defs_ex
# clang++ -Wall -DELEM=float defs_ex.cc -o defs_ex

In [ ]:
%%bash
./defs_ex